## Tasks

4.1 predict magnitude of antibody response - H1N1 A/Victoria/4897/2022 (D28)

4.2 predict magnitude of antibody response - H3N2 A/Massachusetts/18/2022 (D28)

4.3 predict magnitude of antibody response - Vic B/Austria/1359417/2021 (D28)

4.4 predict magnitude of antibody response - all 3 vaccine strains (D28)

**4.5 predict antibody breadth - all variants (D28)**
* Training Data: Demographics + Day 0 + Day 7 innate
* Assay: HAI
* Measure: Geo mean
* Metric: Spearman correlation
* Full description: Geomean HAI across all variants

4.6 predict antibody breadth - all variants (D28)

4.7 predict antibody durability - H1N1 A/Victoria/4897/2022 (D365)

4.8 predict antibody durability - H3N2 A/Massachusetts/18/2022 (D365)

4.9 predict antibody durability - Vic B/Austria/1359417/2021 (D365)

4.10 predict antibody durability - all 3 vaccine strains (D365)

### Task 4.5: Focus
Breadth refers to how widely an antibody response covers different variants of a pathogen (not just the specific strain, but also related versions).
Want to measure a person's antibody response broadly, so we are predicting the 'average protective coverage' across the whole panel of flu variants.
Use the geometric mean (average used for antibody titers)

In [11]:
import numpy as np
import pandas as pd
from scipy.stats import gmean

In [12]:
DATA_PATH = 'data/PART2-26-01-26_reorg/PART2-26-01-26_reorg'
train_hai = pd.read_csv(DATA_PATH + '/train_hai.tsv', sep='\t')
train_participants = pd.read_csv(DATA_PATH + '/train_participants.tsv', sep='\t')

tables = {
    'train_hai': train_hai,
    'train_participants': train_participants,
}

In [13]:
for name, df in tables.items():
    print(f"\n{'=' * 50}")
    print(f'TABLE: {name}')
    print(f"{'=' * 50}")
    display(df.head(5))


TABLE: train_hai


,hai_id,participant_id,timepoint,virus_strain,value,material
0,ID_001__2016_UGA_Standard_Fluzone__0__HAI__H1N...,2016_UGA.ID_001,0.0,H1N1 A/South Carolina/1/1918,20.,Unknown
1,ID_001__2016_UGA_Standard_Fluzone__21__HAI__H1...,2016_UGA.ID_001,28.0,H1N1 A/South Carolina/1/1918,40.,Unknown
2,ID_002__2016_UGA_Standard_Fluzone__0__HAI__H1N...,2016_UGA.ID_002,0.0,H1N1 A/South Carolina/1/1918,5.,Unknown
3,ID_002__2016_UGA_Standard_Fluzone__21__HAI__H1...,2016_UGA.ID_002,28.0,H1N1 A/South Carolina/1/1918,5.,Unknown
4,ID_003__2016_UGA_Standard_Fluzone__0__HAI__H1N...,2016_UGA.ID_003,0.0,H1N1 A/South Carolina/1/1918,80.,Unknown



TABLE: train_participants


,participant_id,subject,biological_sex,race,min_age,max_age,geolocation,investigation_id,investigation_name,arm_id,arm_name,data_source,description,basic_curation,pubmed_ids,main_pmid,main_publication_author
0,SDY269.SUB112836,SUB112836,female,White,28,28,US: Georgia,SDY269,Systems Biology of 2008 Influenza Vaccination ...,ARM1888,LAIV group 2008,"ImmPort, ImmuneSpace",Healthy adults given 2008 LAIV vaccine,no,21743478 26682988,21743478,Nakaya HI (2011)
1,SDY269.SUB112849,SUB112849,female,Black or African American,39,39,US: Georgia,SDY269,Systems Biology of 2008 Influenza Vaccination ...,ARM1888,LAIV group 2008,"ImmPort, ImmuneSpace",Healthy adults given 2008 LAIV vaccine,no,21743478 26682988,21743478,Nakaya HI (2011)
2,SDY269.SUB112854,SUB112854,male,Black or African American,46,46,US: Georgia,SDY269,Systems Biology of 2008 Influenza Vaccination ...,ARM1888,LAIV group 2008,"ImmPort, ImmuneSpace",Healthy adults given 2008 LAIV vaccine,no,21743478 26682988,21743478,Nakaya HI (2011)
3,SDY269.SUB112860,SUB112860,female,White,32,32,US: Georgia,SDY269,Systems Biology of 2008 Influenza Vaccination ...,ARM1888,LAIV group 2008,"ImmPort, ImmuneSpace",Healthy adults given 2008 LAIV vaccine,no,21743478 26682988,21743478,Nakaya HI (2011)
4,SDY269.SUB112881,SUB112881,female,Black or African American,29,29,US: Georgia,SDY269,Systems Biology of 2008 Influenza Vaccination ...,ARM1888,LAIV group 2008,"ImmPort, ImmuneSpace",Healthy adults given 2008 LAIV vaccine,no,21743478 26682988,21743478,Nakaya HI (2011)


#### Data Cleaning
HAI
* Timepoint column only missing 1.89% of values, and value column missing 0.02% of values. We decide to drop these rows.

Participants
* main_publication_author missing 47.55% of values, geolocation missing 20.18%, pubmed_ids missing 3.06%, main_pmid missing 3.06%

In [14]:
# For HAI we can dropna as the amount of missing values is low. Want to make sure value is numeric.
train_hai.dropna()
train_hai['value'] = pd.to_numeric(train_hai['value'], errors='coerce')
# For Participants drop publication column all together
train_participants.drop(columns=['pubmed_ids'], inplace=True)

In [15]:
train_merged = train_hai.merge(train_participants, on='participant_id', how='left')

cols_to_keep = [
    'hai_id', 'participant_id', 'timepoint', 'virus_strain', 'value', 'biological_sex', 'race', 'min_age',
    'geolocation', 'investigation_id', 'investigation_name', 'arm_id', 'arm_name', 'data_source', 'description',
]

train_merged = train_merged[cols_to_keep]
train_merged.dropna()
train_merged.head()

,hai_id,participant_id,timepoint,virus_strain,value,biological_sex,race,min_age,geolocation,investigation_id,investigation_name,arm_id,arm_name,data_source,description
0,ID_001__2016_UGA_Standard_Fluzone__0__HAI__H1N...,2016_UGA.ID_001,0.0,H1N1 A/South Carolina/1/1918,20.,female,race: unknown,29,US: Georgia,2016_UGA,2016_UGA_Fluzone,2016_UGA_Standard_Fluzone,2016 UGA Standard Fluzone,UGA,2016 UGA Standard Fluzone
1,ID_001__2016_UGA_Standard_Fluzone__21__HAI__H1...,2016_UGA.ID_001,28.0,H1N1 A/South Carolina/1/1918,40.,female,race: unknown,29,US: Georgia,2016_UGA,2016_UGA_Fluzone,2016_UGA_Standard_Fluzone,2016 UGA Standard Fluzone,UGA,2016 UGA Standard Fluzone
2,ID_002__2016_UGA_Standard_Fluzone__0__HAI__H1N...,2016_UGA.ID_002,0.0,H1N1 A/South Carolina/1/1918,5.,female,race: unknown,29,US: Georgia,2016_UGA,2016_UGA_Fluzone,2016_UGA_Standard_Fluzone,2016 UGA Standard Fluzone,UGA,2016 UGA Standard Fluzone
3,ID_002__2016_UGA_Standard_Fluzone__21__HAI__H1...,2016_UGA.ID_002,28.0,H1N1 A/South Carolina/1/1918,5.,female,race: unknown,29,US: Georgia,2016_UGA,2016_UGA_Fluzone,2016_UGA_Standard_Fluzone,2016 UGA Standard Fluzone,UGA,2016 UGA Standard Fluzone
4,ID_003__2016_UGA_Standard_Fluzone__0__HAI__H1N...,2016_UGA.ID_003,0.0,H1N1 A/South Carolina/1/1918,80.,female,race: unknown,28,US: Georgia,2016_UGA,2016_UGA_Fluzone,2016_UGA_Standard_Fluzone,2016 UGA Standard Fluzone,UGA,2016 UGA Standard Fluzone


In [16]:
# Count occurrences of each timepoint
timepoint_counts = train_merged['timepoint'].value_counts()
print(timepoint_counts)

timepoint
 0.0      49090
 28.0     48312
 365.0    23808
 90.0      2202
 14.0       777
 30.0       468
 3.0        318
 75.0       318
 7.0        209
 70.0       180
 180.0       51
-7.0         18
Name: count, dtype: int64


In [17]:
# 7.0 has very few data points, so we do not use it
train_merged = train_merged[train_merged['timepoint'].isin([0.0, 28.0])]

### Build the target variable (y) - Geomean of HAI at Day 28

In [18]:
timepoint_counts = train_merged['timepoint'].value_counts()
print(timepoint_counts)

timepoint
0.0     49090
28.0    48312
Name: count, dtype: int64


In [19]:
d28 = train_merged[train_merged['timepoint'] == 28.0].copy()
d28['value'] = pd.to_numeric(d28['value'], errors='coerce')
y = (
    d28.groupby('participant_id')['value']
    .apply(lambda x: gmean(x))
    .reset_index()
    .rename(columns={'value': 'geomean_d28'})
)

In [20]:
y

,participant_id,geomean_d28
0,2016_UGA.ID_001,122.696422
1,2016_UGA.ID_002,42.430638
2,2016_UGA.ID_003,132.085872
3,2016_UGA.ID_004,33.021468
4,2016_UGA.ID_005,43.061034
...,...,...
3626,SDY887.SUB134259,512.000000
3627,SDY887.SUB134260,161.269894
3628,SDY887.SUB197783,25.398417
3629,SDY887.SUB197784,101.593667


### Build the features (X) - Day 0 of HAI and demographics

In [20]:
# Filter to Day 0 — the baseline readings before vaccination
d0 = train_merged[train_merged['timepoint'] == 0.0].copy()
d0['value'] = pd.to_numeric(d0['value'], errors='coerce')

# Pivot: one row per participant, one column per virus strain
# So instead of long format (many rows per person), we get wide format (one row per person)
d0_wide = d0.pivot_table(
    index='participant_id',
    columns='virus_strain',
    values='value',
    aggfunc='mean'  # if duplicate entries exist, average them
).reset_index()

# Log-transform the HAI columns (standard practice — brings them to a linear scale)
hai_cols = [c for c in d0_wide.columns if c != 'participant_id']
d0_wide[hai_cols] = np.log1p(d0_wide[hai_cols])  # log1p = log(1+x), handles near-zero values

# Get demographics — one row per participant (drop duplicates since each person appears many times)
demo = train_merged[['participant_id', 'biological_sex', 'race', 'min_age']].drop_duplicates()

# Encode categorical variables — models need numbers, not text
demo = pd.get_dummies(demo, columns=['biological_sex', 'race'], drop_first=True)

# Merge Day 0 HAI + demographics into one feature table
X_df = d0_wide.merge(demo, on='participant_id', how='inner')